# Physics-Informed Graph Neural Network for Storm Surge (Cyclone Amphan)
This notebook trains a **True Autoregressive PI-GNN Simulator** using Parametric Wind Physics (Holland 1980) and Explicit Euler Integration, mathematically mirroring ADCIRC.

**IMPORTANT:** Make sure you have the **GPU T4 x2** accelerator enabled in the Kaggle session settings before running!

In [ ]:
# 1. Download the updated code and data from GitHub
!git clone https://github.com/ATIK2110018/storm_surge_PI_GNN.git
%cd storm_surge_PI_GNN

# 2. Install required PyTorch Geometric and NetCDF libraries
!pip install torch-geometric netCDF4 matplotlib

## 1. Pre-Training Visualization (Physics Inputs)
Visualizing the ADCIRC mesh, open boundaries, storm track, and the **Holland (1980)** generated parametric wind fields.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import numpy as np
import sys

sys.path.append('PI-GNN')
from dataset.process_adcirc import load_adcirc_mesh, parse_fort22, holland_wind_model

print("=== PI-GNN Domain & Physics Forcing Visualization ===")
f14 = 'model_io/fort.14'
f22 = 'model_io/fort.22'

print("Loading Mesh, Boundaries, and Track...")
nodes, elements, open_boundary_nodes = load_adcirc_mesh(f14)
track_data = parse_fort22(f22)

x, y, depth = nodes[:, 0], nodes[:, 1], nodes[:, 2]
triangulation = tri.Triangulation(x, y, elements)

track_lons = [pt['lon'] for pt in track_data]
track_lats = [pt['lat'] for pt in track_data]

# PLOT 1: Domain
plt.figure(figsize=(12, 10))
plt.title("PI-GNN Domain: Bathymetry, Boundaries & Cyclone Track", fontsize=16)
contour = plt.tricontourf(triangulation, depth, levels=50, cmap='ocean_r')
plt.colorbar(contour, label='Depth (m)')
plt.triplot(triangulation, color='black', alpha=0.1, linewidth=0.1)

if len(open_boundary_nodes) > 0:
    plt.scatter(x[open_boundary_nodes], y[open_boundary_nodes], color='red', s=10, label='Open Ocean Boundary')
    
plt.plot(track_lons, track_lats, color='orange', linewidth=2, linestyle='--', label='Cyclone Track')
plt.scatter(track_lons, track_lats, color='yellow', edgecolor='black', s=30, zorder=5)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(loc='lower right')
plt.show()

# PLOT 2: Holland Physics
print("Generating Analytical Holland Wind/Pressure Fields...")
step = 18
storm = track_data[step]

p_field, u_field, v_field = holland_wind_model(x, y, storm['lon'], storm['lat'], storm['vmax'], storm['pc'])
wind_mag = np.sqrt(u_field**2 + v_field**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

c1 = ax1.tricontourf(triangulation, wind_mag, levels=30, cmap='YlOrRd')
ax1.plot(track_lons, track_lats, color='black', linestyle='--', alpha=0.5)
ax1.scatter([storm['lon']], [storm['lat']], color='red', marker='X', s=200, label="Storm Eye")
ax1.set_title(f"Holland Wind Magnitude (m/s)\nVmax: {storm['vmax']} kts")
fig.colorbar(c1, ax=ax1, label='Wind Speed (m/s)')
ax1.legend()

c2 = ax2.tricontourf(triangulation, p_field, levels=30, cmap='coolwarm')
ax2.plot(track_lons, track_lats, color='black', linestyle='--', alpha=0.5)
ax2.scatter([storm['lon']], [storm['lat']], color='red', marker='X', s=200)
ax2.set_title(f"Holland Atmospheric Pressure (mb)\nPc: {storm['pc']} mb")
fig.colorbar(c2, ax=ax2, label='Pressure (mb)')
plt.show()

## 2. Train the True Autoregressive Simulator
This script generates the physical forcing fields and executes the Autoregressive Euler GNN Simulator for 1000 Epochs.
The loss is calculated purely based on the model's self-generated hydrograph after the 3-day simulation.

In [ ]:
!python PI-GNN/training/train.py

## 3. Water Resources Engineering Validations
Evaluates the trained model against the ADCIRC ground truth using Hydrographs, R^2 Scatter, and Spatial Error maps.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import sys
from sklearn.metrics import mean_squared_error, r2_score

sys.path.append('PI-GNN')
from model.st_gnn import AutoregressiveSurrogate
from dataset.process_adcirc import create_full_simulation_dataset

print("=== PI-GNN Engineering Evaluation ===")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

f14 = 'model_io/fort.14'
f22 = 'model_io/fort.22'
f63 = 'model_io/fort.63.nc'

print("1. Re-loading the test data and trained model...")
forcing_sequence, edge_index, true_zetas, open_boundary_nodes, boundary_tides = create_full_simulation_dataset(f14, f22, f63)

num_nodes = forcing_sequence.size(1)
model = AutoregressiveSurrogate(num_nodes=num_nodes, num_forcing_features=5).to(device)
model_path = 'PI-GNN/training/pi_gnn_model.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("2. Running the Full Autoregressive Simulation...")
forcing_sequence = forcing_sequence.to(device)
edge_index = edge_index.to(device)
boundary_tides = boundary_tides.to(device)

with torch.no_grad():
    simulated_zetas = model(forcing_sequence, edge_index, open_boundary_nodes, boundary_tides)
    
preds_array = simulated_zetas.cpu().numpy().squeeze()
truth_array = true_zetas.numpy().squeeze()

print("3. Generating Engineering Visualizations...")

# 3A: Hydrographs
test_nodes = [1000, 5000, 15000] 
time_axis = np.arange(preds_array.shape[0])

fig, axs = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
fig.suptitle('Storm Surge Hydrographs: ADCIRC vs PI-GNN', fontsize=16)
for i, node_id in enumerate(test_nodes):
    if node_id < num_nodes:
        axs[i].plot(time_axis, truth_array[:, node_id], label='ADCIRC', color='black', linewidth=2)
        axs[i].plot(time_axis, preds_array[:, node_id], label='PI-GNN (Self-Simulated)', color='red', linestyle='--', linewidth=2)
        axs[i].set_ylabel('Elevation (m)')
        axs[i].set_title(f'Coastal Node ID: {node_id}')
        axs[i].legend()
        axs[i].grid(True)
axs[-1].set_xlabel('Time Steps')
plt.tight_layout()
plt.show()

# 3B: Peak Surge Scatter
peak_truth = np.max(truth_array, axis=0)
peak_preds = np.max(preds_array, axis=0)
valid_idx = (peak_truth > 0.1)
peak_truth_valid = peak_truth[valid_idx]
peak_preds_valid = peak_preds[valid_idx]

if len(peak_truth_valid) > 0:
    r2 = r2_score(peak_truth_valid, peak_preds_valid)
    rmse = np.sqrt(mean_squared_error(peak_truth_valid, peak_preds_valid))
    plt.figure(figsize=(8, 8))
    plt.scatter(peak_truth_valid, peak_preds_valid, alpha=0.3, color='blue', s=2)
    plt.plot([0, np.max(peak_truth_valid)], [0, np.max(peak_truth_valid)], 'r--', label='Perfect Agreement (1:1)')
    plt.title(f"Peak Surge Correlation\n$R^2$: {r2:.3f} | RMSE: {rmse:.3f} m")
    plt.xlabel("ADCIRC Peak Surge (m)")
    plt.ylabel("PI-GNN Peak Surge (m)")
    plt.legend()
    plt.grid(True)
    plt.show()
    
# 3C: Spatial Error Map
timestep = 50
if timestep < preds_array.shape[0]:
    spatial_error = np.abs(truth_array[timestep, :] - preds_array[timestep, :])
    
    with open(f14, 'r') as f:
        f.readline()
        ne, nn = map(int, f.readline().split())
        x, y = np.zeros(nn), np.zeros(nn)
        elements = np.zeros((ne, 3), dtype=int)
        for i in range(nn):
            parts = f.readline().split()
            x[i], y[i] = float(parts[1]), float(parts[2])
        for i in range(ne):
            parts = f.readline().split()
            elements[i,0], elements[i,1], elements[i,2] = int(parts[2])-1, int(parts[3])-1, int(parts[4])-1
    
    triangulation = tri.Triangulation(x, y, elements)
    plt.figure(figsize=(12, 10))
    plt.title(f"Absolute Spatial Error (m) at t={timestep}", fontsize=16)
    contour = plt.tricontourf(triangulation, spatial_error, levels=30, cmap='Reds')
    plt.colorbar(contour, label='Absolute Error |ADCIRC - GNN| (m)')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()